In [241]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
from joblib import dump





In [242]:

%store -r data
data=data
%store -r preprocessor
preprocessor=preprocessor



In [243]:
#print(data)

### Define variable

In [244]:
# Prepare the full feature set
# x_content is the feature set including content-based features
X_content = preprocessor.fit_transform(data) # feature vector for al auto
X_content.shape



(1994, 54)

In [245]:
feature_price='Sale price'
feature_milleage='Milleage'
feature_Model='Model'
my_modell = "Focus"

#### **Cosinus similarity**
Cosine Similarity misst:

„Wie ähnlich ist Auto i zu Auto j basierend auf ihren Features?“


In [246]:
cosine_sim = cosine_similarity(X_content, X_content)

In [247]:
car_index = 10
cos_index=cosine_sim[car_index] # similarity scores for the car at index 10
cos_index

array([ 0.1115753 ,  0.25823025,  0.17216766, ...,  0.11549804,
        0.18204823, -0.15954792])

#### **Content-Based**
**iloc** wählt Zeilen per Position
(Index des Autos, Ähnlichkeitswert)

In [248]:
size_cosine_sim=len(cosine_sim)
print(size_cosine_sim)

1994


In [249]:
data['item_id'] = data.index  # jedes Auto bekommt eine eindeutige ID
#print(data['item_id'].head())
#print(data.loc[3])

len(data)

1994

In [250]:
# Function to recommend similar cars based on cosine similarity
def recommend_similar_cars(car_index, top_n=10):
     
    n_items = cosine_sim.shape[0]

    if car_index < 0 or car_index >= n_items:
            raise IndexError(f"car_index must be between 0 and {n_items-1}")
    
    # Get the cosine similarity scores for the item
    similarity_scores = list(enumerate(cosine_sim[car_index]))

    # Sort similar items by similarity score in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Get the indices of the top N similar items (excluding the first one which is the item itself)
    similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

    return data.iloc[similar_indices]


#### content-based by model

In [251]:
# Funktion zur Empfehlung ähnlicher Autos basierend auf Cosinus-Ähnlichkeit
def recommend_contentbased_by_model(model_name, top_n=5):
    try:    # Prüfen, ob das Modell im Datensatz existiert
        if model_name not in data[feature_Model].values:
            raise ValueError(f"Modell '{model_name}' nicht im Datensatz gefunden.")

        # Index des Autos anhand des Modellnamens
        car_index = data.index[data[feature_Model] == model_name][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        return data.iloc[similar_indices]

    except ValueError as e:
        print(e)
        return pd.DataFrame()



In [252]:
#contentbased_by_model=recommend_contentbased_by_model(model_name, top_n=5)
contentbased_by_model=recommend_contentbased_by_model('Focus')
print(contentbased_by_model)


        Brand   Model  year_of_manufacture  Engine_power(kilowatt)  \
1561  Renault  Megane                 2017                     170   
1043     Ford   Focus                 2021                      86   
197   Hyundai     i30                 2018                     127   
1147     Ford  Fiesta                 2019                     145   
263   Hyundai    Kona                 2020                     105   

      Engine_power_(Horsepower) Transmission_type  Mileage Fueltype  \
1561                        231         automatic   198632  Elektro   
1043                        116            manual   133454   Diesel   
197                         172            manual   202753  Elektro   
1147                        197         automatic    34109   Hybrid   
263                         142         automatic   108562  Elektro   

      Nber_previous_owners Body_style  ... damaged_roof_beam  \
1561                     3        suv  ...                 1   
1043                    

#### **Contend based by price**

In [253]:
def recommend_similar_cars_by_price(price, top_n=10):
    try:
        # Prüfen, ob mindestens ein Auto mit dem Preis existiert
        if price not in data[feature_price].values:
            # Falls exakter Preis nicht existiert, nächstgelegenen Preis nehmen
            closest_price = data[feature_price].iloc[(data[feature_price] - price).abs().argsort()[0]]
            print(f"Kein Auto mit Preis {price} gefunden. Nächstgelegener Preis: {closest_price}")
            price = closest_price

        # Index des Autos mit dem Preis
        car_index = data.index[data[feature_price] == price][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        return data.iloc[similar_indices]

    except Exception as e:
        print(f"Fehler: {e}")
        return pd.DataFrame()  # Leere DataFrame zurückgeben


In [290]:
recommend_similar_cars_by_price(1000, top_n=10)

Kein Auto mit Preis 1000 gefunden. Nächstgelegener Preis: 3000


,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,Body_style,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
1310,Mercedes,A-Class,2021,132,179,automatic,194990,Hybrid,3,hatchback,...,1,0,1,1,1,1,A537,708.0,3078,1310
1862,Volkswagen,Passat,2022,182,247,automatic,215450,Hybrid,3,wagon,...,1,0,1,0,1,1,A1603,756.0,6085,1862
188,Mercedes,C-Class,2017,88,119,automatic,161431,Diesel,3,coupe,...,1,0,0,1,1,0,A59,529.0,3000,188
1920,Mercedes,A-Class,2022,117,159,automatic,95122,Diesel,3,coupe,...,1,0,1,0,1,0,A723,191.0,12047,1920
1701,Mercedes,A-Class,2018,114,154,automatic,229706,Benzin,3,hatchback,...,1,1,0,1,1,1,A1776,161.0,3000,1701
1344,Mercedes,A-Class,2021,70,95,automatic,41034,Hybrid,2,hatchback,...,0,0,1,0,0,1,A676,286.0,10376,1344
1220,Volkswagen,Golf,2023,82,111,automatic,92968,Elektro,3,hatchback,...,1,0,0,1,1,1,A1104,883.0,8315,1220
1146,Volkswagen,Passat,2019,67,91,automatic,188666,Elektro,2,coupe,...,1,0,0,0,0,1,A554,763.0,3000,1146
1716,Mercedes,C-Class,2023,100,135,manual,209785,Elektro,3,wagon,...,1,0,1,0,1,1,A2397,661.0,8055,1716
821,Hyundai,Tucson,2017,70,95,automatic,54355,Elektro,3,hatchback,...,1,0,0,0,0,1,A1818,229.0,3000,821


##### **Precision@10 bei Preis**

In [254]:
def relevant_in_top_10_function(recommended_df,target_price, feature_price, tolerance):

    """  recommended_df: Das von Ihrer Funktion zurückgegebene DataFrame (Top 10)
    target_price: Der gesuchte Preis (Wunschpreis)
    feature_price: Der Name Ihrer Preisspalte im DataFrame (z.B. 'price')
    tolerance: Erlaubte Abweichung (0.15 = +/- 15%)
    """
     # Falls ein Fehler auftrat und das DataFrame leer ist
    if recommended_df.empty:
        return 0.0, 0.0
    
    # Top 10 Empfehlungen begrenzen
    top_10 = recommended_df.head(10)
    
    # Grenzen für Relevanz festlegen
    lower_bound = target_price * (1 - tolerance)
    upper_bound = target_price * (1 + tolerance)
    print(f"Relevante Autos im Preisbereich {lower_bound:.2f} - {upper_bound:.2f}:")

    # Prüfen, welche Autos im relevanten Preisbereich liegen
    # Gibt eine Serie von True/False zurück
    relevant_in_top_10 = (top_10[feature_price] >= lower_bound) & (top_10[feature_price] <= upper_bound)
    print(top_10[relevant_in_top_10][[feature_price, feature_Model]])  # Zeige die relevanten Autos an
    # Anzahl der True-Werte zählen
    relevant_count = relevant_in_top_10.sum()
    print(f"{relevant_count} von 10 Empfehlungen sind im relevanten Preisbereich.")
    
    # 1. Relevante Autos in den Top 10 zählen (für Precision & Recall)
    relevant_in_top_10 = ((top_10[feature_price] >= lower_bound) & (top_10[feature_price] <= upper_bound)).sum()
    

    return relevant_count

In [255]:

def evaluate_contend_based_precision_at_10(recommended_df, target_price, feature_price, tolerance=0.15):
    """
    Berechnet ausschließlich die Precision@10 für die Empfehlungsliste.
    """
    
    relevant_in_top_10 =relevant_in_top_10_function(recommended_df,target_price,feature_price,tolerance)
    return relevant_in_top_10 / 10

In [289]:
wunschpreis = 4500
# use the actual price column name defined earlier
name_der_preisspalte = feature_price  # 'Sale price'

# 1. Empfehlungen generieren
empfehlungen = recommend_similar_cars_by_price(wunschpreis, top_n=10)

# 2. Precision@10 berechnen
p10 = evaluate_contend_based_precision_at_10(empfehlungen, wunschpreis, name_der_preisspalte, tolerance=0.15)

print(f"Precision@10 für Preis {wunschpreis}: {p10:.2f} (Das sind {p10*100:.0f}% relevante Autos)")

Kein Auto mit Preis 4500 gefunden. Nächstgelegener Preis: 4499
Relevante Autos im Preisbereich 3825.00 - 5175.00:
     Sale price    Model
631        4059  A-Class
1 von 10 Empfehlungen sind im relevanten Preisbereich.
Precision@10 für Preis 4500: 0.10 (Das sind 10% relevante Autos)


#### **Recall@10**

In [257]:
def evaluate_precision_and_recall_at_10(recommended_df, target_price, feature_price, full_data, tolerance=0.15):
    """
    Berechnet Precision@10 und Recall@10 für eine Empfehlungsliste.
    
    recommended_df: Das von Ihrer Funktion zurückgegebene DataFrame (Top 10)
    target_price: Der gesuchte Preis (Wunschpreis)
    feature_price: Der Name Ihrer Preisspalte im DataFrame
    full_data: Das gesamte ursprüngliche DataFrame (wird für den Recall benötigt)
    tolerance: Erlaubte Abweichung (0.15 = +/- 15%)
    """
    # Falls ein Fehler auftrat und das DataFrame leer ist
    if recommended_df.empty:
        return 0.0, 0.0
    
    # Top 10 Empfehlungen begrenzen
    top_10 = recommended_df.head(10)
    
    # Grenzen für Relevanz festlegen
    lower_bound = target_price * (1 - tolerance)
    upper_bound = target_price * (1 + tolerance)
    
    # 1. Relevante Autos in den Top 10 zählen (für Precision & Recall)
    relevant_in_top_10 = ((top_10[feature_price] >= lower_bound) & 
                          (top_10[feature_price] <= upper_bound)).sum()
    
    # 2. Gesamtzahl aller existierenden relevanten Autos in der Datenbank zählen (für Recall)
    total_relevant_in_db = ((full_data[feature_price] >= lower_bound) & 
                            (full_data[feature_price] <= upper_bound)).sum()
    
    # 3. Precision@10 berechnen
    precision_10 = relevant_in_top_10 / 10
    
    # 4. Recall@10 berechnen (Sonderfall abfangen, falls gar kein Auto in der DB passt)
    if total_relevant_in_db > 0:
        recall_10 = relevant_in_top_10 / total_relevant_in_db
    else:
        recall_10 = 0.0
        
    return precision_10, recall_10


In [258]:
wunschpreis = 15000

# 1. Empfehlungen generieren
empfehlungen = recommend_similar_cars_by_price(wunschpreis, top_n=10)

# 2. Beide Metriken berechnen (wir übergeben 'data' als gesamte Datenbank)
p10, r10 = evaluate_precision_and_recall_at_10(
    recommended_df=empfehlungen, 
    target_price=wunschpreis, 
    feature_price=feature_price, 
    full_data=data, 
    tolerance=0.15
)

# 3. Ergebnisse ausgeben
print(f"Ergebnisse für Wunschpreis {wunschpreis} €:")
print(f"- Precision@10: {p10:.2f} ({p10*100:.0f}% der Empfehlungen passen preislich)")
print(f"- Recall@10:    {r10:.2f} ({r10*100:.0f}% aller passenden Autos aus der DB wurden gefunden)")


Kein Auto mit Preis 15000 gefunden. Nächstgelegener Preis: 14979
Ergebnisse für Wunschpreis 15000 €:
- Precision@10: 0.10 (10% der Empfehlungen passen preislich)
- Recall@10:    0.01 (1% aller passenden Autos aus der DB wurden gefunden)


#### Milege contend based

In [259]:
def recommend_contendbased_by_milleage(km, top_n=10):
    try:
        # Prüfen, ob mindestens ein Auto mit dem Kilometerstand existiert
        if km not in data[feature_milleage].values:
            # Falls exakter Kilometerstand nicht existiert, nächstgelegenen Wert nehmen
            closest_km = data[feature_milleage].iloc[(data[feature_milleage] - km).abs().argsort()[0]]
            print(f"Kein Auto mit {km} km gefunden. Nächstgelegener Kilometerstand: {closest_km}")
            km = closest_km

        # Index des Autos mit dem Kilometerstand
        car_index = data.index[data[feature_milleage] == km][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        return data.iloc[similar_indices]

    except Exception as e:
        print(f"Fehler: {e}")
        return pd.DataFrame()  # Leere DataFrame zurückgeben


In [260]:
def recommend_contendbased(car_index, top_n=5):
    try:
        recommended_cars = recommend_similar_cars(car_index, top_n=top_n)
        
    except IndexError as e:
        print("Fehler:", e)
        recommended_cars = pd.DataFrame()  # leeres DataFrame als Fallback

    return recommended_cars

In [261]:
recommend_contendbased(5, top_n=5)
#recommend_similar_cars(6, top_n=5)


,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,Body_style,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
1261,Hyundai,Kona,2023,212,288,manual,222101,Elektro,2,suv,...,0,1,1,1,0,0,A720,349.0,10377,1261
540,Mercedes,C-Class,2023,77,104,automatic,215022,Benzin,1,sedan,...,0,1,0,1,1,0,A2355,895.0,7335,540
1303,Mercedes,C-Class,2018,152,206,automatic,226221,Diesel,2,wagon,...,0,1,0,1,0,0,A697,465.0,3000,1303
1417,Mercedes,GLA,2023,117,159,manual,93688,Hybrid,2,sedan,...,1,1,0,1,0,0,A918,589.0,11183,1417
715,Renault,Captur,2023,117,159,manual,67536,Elektro,3,suv,...,0,0,0,1,0,0,A1927,386.0,12052,715


In [262]:
data.columns = data.columns.str.strip()

#data[['Brand','Model', 'images']]  # funktioniert jetzt


#### **Collaborative Filtering**
Empfiehlt Items basierend auf dem Verhalten vieler Nutzer, nicht auf Item-Features.

Item-Item Cosine Similarity berechnen

#### **Build User_Matrix**

user_item_matrix: Zeilen = Nutzer, Spalten = Autos, Werte = Ratings

In [263]:


data['item_id'] = data.index  # eindeutige ID für jedes Auto

# Simulierte User erstellen
n_users = 5  # number of user
n_items = len(data)

# ratings array
ratings_list = []

np.random.seed(42)  # für Reproduzierbarkeit

#Jeder User bewertet jedes Auto
for user_id in range(1, n_users + 1):
    for item_id in data['item_id']:

        # Zufällige Bewertung, z.B. 1-5
        rating = np.random.randint(1, 6)
        ratings_list.append({
            'user_id': user_id,
            'item_id': item_id,
            'rating': rating
        })

# Ratings DataFrame erstellen
ratings_df = pd.DataFrame(ratings_list)

# User–Item-Matrix erstellen
user_item_matrix = ratings_df.pivot_table(
    index='user_id',      # Zeilen = User
    columns='item_id',    # Spalten = Items
    values='rating',      # Werte = Ratings
    fill_value=0          # keine Bewertung = 0
)




In [264]:
#data['item_id'] # eindeutige ID für jedes Auto
'item_id' in data.columns
type(data['item_id'])# pandas.core.series.Series
data[['item_id']].columns



Index(['item_id'], dtype='object')

In [265]:
ratings_df

,user_id,item_id,rating
0,1,0,4
1,1,1,5
2,1,2,3
3,1,3,5
4,1,4,5
...,...,...,...
9965,5,1989,1
9966,5,1990,2
9967,5,1991,3
9968,5,1992,5


In [266]:

cosine_sim_item = cosine_similarity(user_item_matrix.T)
cosine_sim_item.shape

(1994, 1994)

In [267]:
#print(user_item_matrix.head(2))

In [268]:
def recommend_collaborativ_cars(car_index, top_n=5):
    """
    Gibt die Top-N ähnlichen Autos zurück basierend auf Features (Item-Based CF)
    """
    n_items = cosine_sim.shape[0]
    
    if car_index < 0 or car_index >= n_items:
        raise ValueError(f"car_index muss zwischen 0 und {n_items-1} liegen")
    
    # Ähnlichkeiten abrufen
    similarity_scores = list(enumerate(cosine_sim_item[car_index]))
    
    # Sortieren nach Ähnlichkeit (absteigend)
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    # Top-N ähnliche Autos, eigenes Auto ausschließen
    similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]
    
    # DataFrame zurückgeben
    return data.iloc[similar_indices]


In [269]:

def recommend_collaborative_by_models(model_name, top_n=5):

    try:    # Prüfen, ob das Modell im Datensatz existiert
        if model_name not in data[feature_Model].astype(str).values:
             raise ValueError(f"Modell '{model_name}' nicht im Datensatz gefunden.")
       

        # Index des Autos anhand des Modellnamens
        car_index = data.index[data[feature_Model] == model_name][0]

        # Cosinus-Ähnlichkeitswerte für das Auto
        similarity_scores = list(enumerate(cosine_sim_item[car_index]))

        # Sortieren nach Ähnlichkeit absteigend
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

        # Indizes der Top-N ähnlichen Autos (ohne das Auto selbst)
        similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

        recommended_car = data.iloc[similar_indices]

        return recommended_car

    except ValueError as e:
        print(e)
        return []

In [270]:
recommend_collaborative_by_models('Focus')

,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,Body_style,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
1166,Renault,Megane,2021,200,271,automatic,24848,Hybrid,3,sedan,...,1,0,1,0,1,0,A2234,67.0,14328,1166
768,Ford,Focus,2023,175,237,automatic,36834,Elektro,1,wagon,...,0,0,0,0,0,1,A591,405.0,17072,768
1858,Ford,Kuga,2020,186,252,manual,143945,Diesel,2,sedan,...,0,1,0,1,0,1,A1623,598.0,6550,1858
17,Mercedes,A-Class,2018,196,266,automatic,247460,Elektro,1,hatchback,...,0,1,0,0,1,0,A1276,476.0,3000,17
1406,Ford,Focus,2017,158,214,manual,213792,Benzin,1,coupe,...,0,0,1,1,0,0,A2461,547.0,3000,1406


In [271]:
# Index des Autos, das empfohlen werden soll
car_index = 1  # z.B. erstes Auto in df

recommended= recommend_collaborativ_cars(car_index, top_n=5)
#print(recommended[['Brand','Model','year of manufacture']])
print(recommended)



         Brand    Model  year_of_manufacture  Engine_power(kilowatt)  \
586   Mercedes      GLA                 2023                     139   
1696   Hyundai   Tucson                 2022                     141   
809    Renault     Clio                 2019                     103   
841   Mercedes      GLA                 2017                      92   
851   Mercedes  A-Class                 2021                     137   

      Engine_power_(Horsepower) Transmission_type  Mileage Fueltype  \
586                         188            manual    65840   Benzin   
1696                        191            manual   193387  Elektro   
809                         140         automatic   183505  Elektro   
841                         125            manual    16713  Elektro   
851                         186            manual   246286   Diesel   

      Nber_previous_owners Body_style  ... damaged_roof_beam  \
586                      2  hatchback  ...                 1   
1696        

#### Hybrid Filtering

In [272]:
def recommend_hybrid_by_models(model_name, top_n=5, alpha=0.5):

    try:
        if model_name not in data[feature_Model].astype(str).values:
            raise ValueError(f"Modell '{model_name}' nicht im Datensatz gefunden.")

        # Item-ID holen
        car_id = data.loc[data[feature_Model] == model_name, 'item_id'].values[0]

        hybrid_scores = []

        for i in range(len(data)):

            if i == car_id:
                continue

            content_score = cosine_sim[car_id][i]
            collab_score = cosine_sim_item[car_id][i]

            hybrid_score = alpha * content_score + (1 - alpha) * collab_score

            hybrid_scores.append((i, hybrid_score))

        # Sortieren
        hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)

        # Top N
        top_indices = [i for i, _ in hybrid_scores[:top_n]]

        recommended_car = data[data['item_id'].isin(top_indices)]

        return (recommended_car)

    except ValueError as e:
        print(e)
        return []

In [273]:
recommend_hybrid_by_models("Focus", top_n=5)

,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,Body_style,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
630,Renault,Megane,2021,179,243,manual,95371,Elektro,2,hatchback,...,1,1,1,0,0,0,A2006,265.0,9873,630
948,Volkswagen,Golf,2019,215,292,manual,156218,Benzin,2,suv,...,1,1,1,0,0,0,A177,884.0,5535,948
1359,Volkswagen,Golf,2017,187,254,manual,173510,Benzin,3,sedan,...,1,1,1,0,0,1,A730,772.0,3184,1359
1926,Renault,Captur,2018,170,231,automatic,207630,Benzin,3,hatchback,...,1,1,0,0,0,0,A2303,104.0,3520,1926
1931,Volkswagen,Tiguan,2019,75,101,manual,238685,Benzin,3,coupe,...,1,1,1,0,0,1,A2277,-109.0,3000,1931


In [274]:
def recommend_hybrid(car_index, top_n=5):
    """
    Hybrid Recommendation:
    alpha = Gewicht für Collaborative Filtering (0-1)
    (1-alpha) = Gewicht für Content-Based
    """
    # Content-Based Score
    contentFiltering_scores = recommend_contendbased(car_index, top_n=top_n)
    
    # CF Score
    collaborative_scores = recommend_collaborativ_cars(car_index, top_n=top_n)

   # merge scores
    hybrid_rec = pd.concat(
      [
        contentFiltering_scores,
        collaborative_scores]
       
    ).drop_duplicates()
    
    return hybrid_rec.head(top_n)


In [275]:
recommend_hybrid(0, top_n=5)

,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,Body_style,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
331,Mercedes,A-Class,2018,85,115,automatic,208859,Benzin,2,sedan,...,1,1,0,0,0,0,A233,-102.0,3000,331
1502,Volkswagen,Passat,2022,62,84,automatic,208261,Benzin,1,coupe,...,1,1,0,1,0,1,A2327,-45.0,3000,1502
1915,Volkswagen,Golf,2019,138,187,automatic,222694,Benzin,1,hatchback,...,1,1,0,0,0,1,A2022,338.0,3000,1915
84,Ford,Focus,2023,142,193,automatic,159461,Elektro,2,suv,...,0,1,0,1,0,0,A2155,-109.0,6319,84
1237,Volkswagen,Golf,2023,93,126,manual,197245,Hybrid,2,suv,...,1,1,0,0,1,0,A2168,212.0,7760,1237


In [276]:
data['images'] = data['images'].str.strip().str.lstrip('/')
(data['images'].head(5))


0    A1511
1    A1613
2    A2268
3     A761
4     A833
Name: images, dtype: object

In [277]:
from pathlib import Path
import streamlit as st



BASE_DIR = Path("../Data/usecar_image/")

def load_all_car_images():

   # folder_path = BASE_DIR / folder_name

    car_images = {}

    for car_folder in BASE_DIR.iterdir():
        if car_folder.is_dir():
            car_id = car_folder.name  # z.B. A1
            
            images = [
                str(img)
                for img in car_folder.glob("*")
                if img.suffix.lower() in ['.png', '.jpg', '.jpeg']
            ]

            car_images[car_id] = images

    return car_images


car_images_dict = load_all_car_images()
#data['image_folder'] = data['images'].str.extract(r'(A\d+)')
#data['image_folder'] = data['images'].apply(lambda x: Path(x[0]).parts[-2] if x else None)

#data['image_files'] = data['image_folder'].map(car_images_dict)
#data['image_files'].head(5)
#print(data['images'].head())

# # --- Streamlit UI ---
# st.title("Car Gallery with Features")

# # Interaktive Auswahl: Auto auswählen
# selected_car = st.selectbox("Wähle ein Auto", data['image_folder'].dropna().unique())

# row = data[data['image_folder'] == selected_car].iloc[0]

# # Auto-Features anzeigen
# st.subheader(f"{row['image_folder']} - {row['Model']} ({row['Brand']})")

# Bilder anzeigen
# if row['images']:
#     for img_path in row['images']:
#         st.image(img_path, width=300)
# else:
#     st.write("Keine Bilder vorhanden")

# for car, images in car_images_dict.items():
#     print(f"\nAuto: {car}")
#     for img in images:
#         print("  ", img)




In [278]:
# data['image_files']
# for idx, row in data.iterrows():
#     print("\nAuto:", row['Brand'], row['Model'])
#     print("Bilder:", row['image_files'])



In [279]:
BASE_DIR = Path("../Data/usecar_image")

def get_images(folder_name):
    folder_path = BASE_DIR / folder_name
    print("Checking:", folder_path)  # DEBUG

    if not folder_path.exists():
        return []
    
    return [
        str(img)
        for img in folder_path.glob("*")
        if img.suffix.lower() in ['.png', '.jpg', '.jpeg']
    ]

#data['image_files'] = data['images'].apply(get_images)



#### Save Variable and function

In [280]:
   
def recommend_hybrid_by_budget(budget, top_n=15, alpha=0.5, price_weight=0.2):
    """
    Hybrid Recommendation mit Budget

    budget        : verfügbares Budget
    top_n         : Anzahl Empfehlungen
    alpha         : Gewicht Collaborative (0-1)
    price_weight  : Gewicht Preisnähe im Ranking
    """

    try:
                   
       # Budget sicher als float
        budget = float(budget)
        #  Autos innerhalb Budget (z.B. +/- 20%)
        lower_bound = budget * 0.8
        upper_bound = budget * 1.2

        budget_cars = data[
            (data['Sale price'] >= lower_bound) &
            (data['Sale price'] <= upper_bound)
        ].copy()

        if budget_cars.empty:
            print("Keine Autos im Budgetbereich gefunden.")
            return []

        results = []

        # Für jedes Auto im Budget Hybrid Score berechnen
        for _, row in budget_cars.iterrows():
            car_id = row['item_id']

            content_score = cosine_sim[car_id].mean()
            collab_score = cosine_sim_item[car_id].mean()

            hybrid_score = (
                (1 - alpha) * content_score +
                alpha * collab_score
            )

            # Preisnähe berechnen (je näher am Budget, desto besser)
            price_score = 1 - abs(row['Sale price'] - budget) / budget

            # Finaler Score
            final_score = hybrid_score + price_weight * price_score

            results.append((car_id, final_score))

        # Sortieren
        results = sorted(results, key=lambda x: x[1], reverse=True)

        top_items = [car_id for car_id, _ in results[:top_n]]

        recommended_cars = data[data['item_id'].isin(top_items)]

        return (recommended_cars)

    except ValueError as e:
        print(e)
        return []
    

In [281]:
recommend_hybrid_by_budget(4000)

,Brand,Model,year_of_manufacture,Engine_power(kilowatt),Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,Body_style,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
103,Hyundai,Kona,2017,116,157,automatic,63544,Hybrid,2,suv,...,1,1,1,0,0,0,A137,312.0,4231,103
248,Volkswagen,Passat,2018,129,175,manual,84418,Diesel,1,suv,...,0,1,1,0,0,0,A1576,738.0,4146,248
294,Mercedes,GLA,2018,107,145,automatic,71480,Hybrid,1,wagon,...,0,0,1,1,0,1,A1885,-20.0,3934,294
555,Hyundai,Kona,2018,153,208,manual,129189,Benzin,3,sedan,...,0,1,0,0,1,0,A539,110.0,3862,555
985,Ford,Focus,2022,112,152,automatic,245680,Elektro,3,hatchback,...,1,1,0,1,0,0,A1237,156.0,4135,985
991,Volkswagen,Passat,2018,186,252,automatic,166083,Benzin,2,hatchback,...,1,0,1,1,1,0,A1080,324.0,4126,991
1006,Ford,Focus,2019,206,280,manual,190625,Benzin,1,wagon,...,1,1,0,1,1,0,A538,714.0,4219,1006
1078,Volkswagen,Passat,2021,107,145,automatic,170330,Elektro,1,coupe,...,1,1,0,0,1,0,A279,-23.0,3976,1078
1358,Renault,Clio,2019,94,127,manual,103146,Elektro,1,suv,...,0,1,1,0,1,1,A1420,588.0,4360,1358
1430,Mercedes,C-Class,2018,172,233,manual,31180,Hybrid,3,coupe,...,0,1,1,1,1,1,A2072,687.0,4312,1430


In [282]:
def recommend_hybrid_by_budgets(budget, top_n=15, alpha=0.5, price_weight=0.2):

    try:
        budget = float(budget)

        lower_bound = budget * 0.8
        upper_bound = budget * 1.2

        budget_cars = data[
            (data['Sale price'] >= lower_bound) &
            (data['Sale price'] <= upper_bound)
        ].copy()

        if budget_cars.empty:
            print("Keine Autos im Budgetbereich gefunden.")
            return []

        scores = []

        for idx, row in budget_cars.iterrows():
            car_id = row['item_id']

            content_score = cosine_sim[car_id].mean()
            collab_score = cosine_sim_item[car_id].mean()

            hybrid_score = (
                (1 - alpha) * content_score +
                alpha * collab_score
            )

            price_score = 1 - abs(row['Sale price'] - budget) / budget

            final_score = hybrid_score + price_weight * price_score

            scores.append(final_score)

        # Score-Spalte hinzufügen
        budget_cars.insert(4, 'scores-wert', scores)
        # Sortieren nach Score
        budget_cars = budget_cars.sort_values(by='scores-wert', ascending=False)
          
        # Top N auswählen
        recommended_cars = budget_cars.head(top_n)

        return recommended_cars

    except ValueError as e:
        print(e)
        return []

In [283]:
recommend_hybrid_by_budgets(budget=4000, top_n=15, alpha=0.5, price_weight=0.2)


,Brand,Model,year_of_manufacture,Engine_power(kilowatt),scores-wert,Engine_power_(Horsepower),Transmission_type,Mileage,Fueltype,Nber_previous_owners,...,damaged_roof_beam,damaged_left_door,damaged_right_door,damaged_seats,interior_dirty,damaged_carosserie,images,days_to_TUV,Sale price,item_id
294,Mercedes,GLA,2018,107,0.697472,145,automatic,71480,Hybrid,1,...,0,0,1,1,0,1,A1885,-20.0,3934,294
1078,Volkswagen,Passat,2021,107,0.692087,145,automatic,170330,Elektro,1,...,1,1,0,0,1,0,A279,-23.0,3976,1078
555,Hyundai,Kona,2018,153,0.690807,208,manual,129189,Benzin,3,...,0,1,0,0,1,0,A539,110.0,3862,555
1878,Ford,Fiesta,2019,117,0.687641,159,automatic,143964,Benzin,1,...,0,1,0,1,0,0,A823,605.0,4007,1878
248,Volkswagen,Passat,2018,129,0.685069,175,manual,84418,Diesel,1,...,0,1,1,0,0,0,A1576,738.0,4146,248
1872,Ford,Kuga,2019,127,0.684788,172,manual,135262,Diesel,3,...,1,0,0,1,1,1,A2406,694.0,4127,1872
103,Hyundai,Kona,2017,116,0.682719,157,automatic,63544,Hybrid,2,...,1,1,1,0,0,0,A137,312.0,4231,103
985,Ford,Focus,2022,112,0.682445,152,automatic,245680,Elektro,3,...,1,1,0,1,0,0,A1237,156.0,4135,985
1006,Ford,Focus,2019,206,0.682000,280,manual,190625,Benzin,1,...,1,1,0,1,1,0,A538,714.0,4219,1006
1589,Renault,Clio,2020,101,0.681648,137,automatic,201538,Hybrid,3,...,0,1,1,0,1,1,A1026,262.0,3806,1589


In [284]:
dump( data, "dataFrame.joblib")


['dataFrame.joblib']

In [285]:
dump( cosine_sim_item, "../recommendsystem/cosine_similarity_collaborativ_filter_item.joblib")


['../recommendsystem/cosine_similarity_collaborativ_filter_item.joblib']

In [286]:
dump( cosine_sim, "cosine_similarity.joblib")



['cosine_similarity.joblib']